In [1]:
import pandas as pd
import mysql.connector

In [2]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",          # change if your username is different
    password="1979",  # put your MySQL password
    database="ev_intelligence_dw"
)
cursor = conn.cursor()
print("Connected successfully")

Connected successfully


In [3]:
iea = pd.read_csv(r"C:\Users\Bala murukan\Downloads\New folder\EV_Intelligencee_platform\02_Cleaned_Data\cleaned_iea_ev_data.csv")
gdp = pd.read_csv(r"C:\Users\Bala murukan\Downloads\New folder\EV_Intelligencee_platform\02_Cleaned_Data\cleaned_gdp.csv")
pop = pd.read_csv(r"C:\Users\Bala murukan\Downloads\New folder\EV_Intelligencee_platform\02_Cleaned_Data\cleaned_population.csv")
elec = pd.read_csv(r"C:\Users\Bala murukan\Downloads\New folder\EV_Intelligencee_platform\02_Cleaned_Data\cleaned_electricity_access.csv")

In [4]:
countries = pd.concat([
    iea[["Country"]],
    gdp[["Country"]],
    pop[["Country"]],
    elec[["Country"]]
]).drop_duplicates().reset_index(drop=True)

countries = countries.rename(columns={"Country": "country_name"})
print("Total unique countries:", len(countries))
countries.head()

Total unique countries: 222


,country_name
0,World
1,China
2,India
3,United States
4,Germany


In [5]:
for index, row in countries.iterrows():
    cursor.execute(
        "INSERT INTO dim_country (country_name) VALUES (%s)",
        (row["country_name"],)
    )

conn.commit()
print("dim_country loaded successfully")

dim_country loaded successfully


In [6]:
cursor.execute("SELECT COUNT(*) FROM dim_country")
print("Total rows in dim_country:", cursor.fetchone()[0])

Total rows in dim_country: 222


In [7]:
# Get all unique years from cleaned files
years = pd.concat([
    iea[["Year"]],
    gdp[["Year"]],
    pop[["Year"]],
    elec[["Year"]]
]).drop_duplicates().sort_values("Year").reset_index(drop=True)

print("Years available:")
print(years)

Years available:
   Year
0  2015
1  2016
2  2017
3  2018
4  2019
5  2020
6  2021
7  2022
8  2023
9  2024


In [8]:
for index, row in years.iterrows():
    cursor.execute(
        "INSERT INTO dim_time (year) VALUES (%s)",
        (int(row["Year"]),)
    )

conn.commit()
print("dim_time loaded successfully")

dim_time loaded successfully


In [9]:
cursor.execute("SELECT * FROM dim_time ORDER BY year")
for row in cursor.fetchall():
    print(row)

(1, 2015, None, None)
(2, 2016, None, None)
(3, 2017, None, None)
(4, 2018, None, None)
(5, 2019, None, None)
(6, 2020, None, None)
(7, 2021, None, None)
(8, 2022, None, None)
(9, 2023, None, None)
(10, 2024, None, None)


In [10]:
company_fin = pd.read_csv(r"C:\Users\Bala murukan\Downloads\New folder\EV_Intelligencee_platform\02_Cleaned_Data\cleaned_company_financials.csv")
stock = pd.read_csv(r"C:\Users\Bala murukan\Downloads\New folder\EV_Intelligencee_platform\02_Cleaned_Data\cleaned_stock_prices.csv")

In [11]:
companies = pd.concat([
    company_fin[["Company", "Ticker"]],
    stock[["Company", "Ticker"]]
]).drop_duplicates().reset_index(drop=True)

companies = companies.rename(columns={
    "Company": "company_name",
    "Ticker": "ticker"
})

print("Total unique companies:", len(companies))
print(companies)

Total unique companies: 34
             company_name         ticker
0                   Tesla           TSLA
1                    Ford              F
2          General Motors             GM
3                  Rivian           RIVN
4                   Lucid           LCID
5                     BYD        1211.HK
6                     NIO            NIO
7                   XPeng           XPEV
8                 Li Auto             LI
9                   Geely        0175.HK
10  Mahindra and Mahindra         M&M.NS
11          Ashok Leyland    ASHOKLEY.NS
12          Eicher Motors   EICHERMOT.NS
13              TVS Motor    TVSMOTOR.NS
14             Bajaj Auto  BAJAJ-AUTO.NS
15          Hero MotoCorp  HEROMOTOCO.NS
16      Olectra Greentech     OLECTRA.NS
17               JBM Auto        JBMA.NS
18           Ola Electric     OLAELEC.NS
19                 Toyota         7203.T
20                  Honda         7267.T
21                 Nissan         7201.T
22                Hyundai     

In [12]:
for index, row in companies.iterrows():
    cursor.execute(
        "INSERT INTO dim_company (company_name, ticker) VALUES (%s, %s)",
        (row["company_name"], row["ticker"])
    )

conn.commit()
print("dim_company loaded successfully")

dim_company loaded successfully


In [13]:
cursor.execute("SELECT COUNT(*) FROM dim_company")
print("Total rows in dim_company:", cursor.fetchone()[0])

Total rows in dim_company: 34


In [15]:
print(iea.columns.tolist())

['Country', 'Category', 'Parameter', 'Mode', 'Powertrain', 'Year', 'Unit', 'Value', 'Aggregate_Group']


In [16]:
powertrains = iea[["Powertrain"]].drop_duplicates().dropna().reset_index(drop=True)
powertrains = powertrains.rename(columns={"Powertrain": "powertrain"})

print(powertrains)

for index, row in powertrains.iterrows():
    cursor.execute(
        "INSERT INTO dim_powertrain (powertrain) VALUES (%s)",
        (row["powertrain"],)
    )

conn.commit()
print("dim_powertrain loaded successfully")

  powertrain
0        BEV
1       PHEV
2       FCEV
dim_powertrain loaded successfully


In [17]:
modes = iea[["Mode"]].drop_duplicates().dropna().reset_index(drop=True)
modes = modes.rename(columns={"Mode": "mode"})

print(modes)

for index, row in modes.iterrows():
    cursor.execute(
        "INSERT INTO dim_mode (mode) VALUES (%s)",
        (row["mode"],)
    )

conn.commit()
print("dim_mode loaded successfully")

               mode
0  2 and 3 wheelers
1              Cars
2              Vans
3             Buses
4            Trucks
dim_mode loaded successfully


In [18]:
categories = iea[["Category"]].drop_duplicates().dropna().reset_index(drop=True)
categories = categories.rename(columns={"Category": "category"})

print(categories)

for index, row in categories.iterrows():
    cursor.execute(
        "INSERT INTO dim_category (category) VALUES (%s)",
        (row["category"],)
    )

conn.commit()
print("dim_category loaded successfully")

     category
0  Historical
dim_category loaded successfully


In [20]:
# Get ID mappings from dimension tables
cursor.execute("SELECT country_id, country_name FROM dim_country")
country_map = {name: id for id, name in cursor.fetchall()}

cursor.execute("SELECT time_id, year FROM dim_time")
time_map = {year: id for id, year in cursor.fetchall()}

cursor.execute("SELECT powertrain_id, powertrain FROM dim_powertrain")
powertrain_map = {name: id for id, name in cursor.fetchall()}

cursor.execute("SELECT mode_id, mode FROM dim_mode")
mode_map = {name: id for id, name in cursor.fetchall()}

cursor.execute("SELECT category_id, category FROM dim_category")
category_map = {name: id for id, name in cursor.fetchall()}

print("Lookup maps created successfully")

Lookup maps created successfully


In [21]:
ev = iea.copy()

# Map foreign keys
ev["country_id"] = ev["Country"].map(country_map)
ev["time_id"] = ev["Year"].map(time_map)
ev["powertrain_id"] = ev["Powertrain"].map(powertrain_map)
ev["mode_id"] = ev["Mode"].map(mode_map)
ev["category_id"] = ev["Category"].map(category_map)

# Keep only needed columns
ev_final = ev[[
    "country_id", "time_id", "powertrain_id", "mode_id", "category_id",
    "Parameter", "Value", "Unit"
]].dropna(subset=["country_id", "time_id"])

print("Rows ready to load:", len(ev_final))
ev_final.head()

Rows ready to load: 5024


,country_id,time_id,powertrain_id,mode_id,category_id,Parameter,Value,Unit
0,1,10,1,1,1,EV stock,79000000.0,Vehicles
1,1,9,1,1,1,EV stock,69000000.0,Vehicles
2,2,10,1,1,1,EV stock,67000000.0,Vehicles
3,2,9,1,1,1,EV stock,60000000.0,Vehicles
4,1,8,1,1,1,EV stock,53000000.0,Vehicles


In [22]:
for index, row in ev_final.iterrows():
    cursor.execute("""
        INSERT INTO fact_ev_metrics 
        (country_id, time_id, powertrain_id, mode_id, category_id, parameter, value, unit)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        int(row["country_id"]),
        int(row["time_id"]),
        int(row["powertrain_id"]) if pd.notna(row["powertrain_id"]) else None,
        int(row["mode_id"]) if pd.notna(row["mode_id"]) else None,
        int(row["category_id"]) if pd.notna(row["category_id"]) else None,
        row["Parameter"],
        float(row["Value"]) if pd.notna(row["Value"]) else None,
        row["Unit"]
    ))

conn.commit()
print("fact_ev_metrics loaded successfully")

fact_ev_metrics loaded successfully


In [23]:
cursor.execute("SELECT COUNT(*) FROM fact_ev_metrics")
print("Total rows in fact_ev_metrics:", cursor.fetchone()[0])

Total rows in fact_ev_metrics: 5024


In [24]:
oil = pd.read_csv(r"C:\Users\Bala murukan\Downloads\New folder\EV_Intelligencee_platform\02_Cleaned_Data\cleaned_crude_oil.csv")
print(oil)

   Year  Crude_Oil_Price
0  2015        50.750000
1  2016        42.808333
2  2017        52.816667
3  2018        68.358333
4  2019        61.416667
5  2020        41.250000
6  2021        69.083333
7  2022        97.100000
8  2023        80.766667
9  2024        78.725000


In [25]:
# Merge GDP, Population, Electricity
eco = gdp.merge(pop[["Country", "Year", "Population"]], on=["Country", "Year"], how="outer")
eco = eco.merge(elec[["Country", "Year", "Access_to_Electricity"]], on=["Country", "Year"], how="outer")

# Add Crude Oil Price (same for all countries in a year)
eco = eco.merge(oil, on="Year", how="left")

# Map foreign keys
eco["country_id"] = eco["Country"].map(country_map)
eco["time_id"] = eco["Year"].map(time_map)

eco_final = eco[[
    "country_id", "time_id", "GDP", "Population", "Access_to_Electricity", "Crude_Oil_Price"
]].dropna(subset=["country_id", "time_id"])

print("Rows ready to load:", len(eco_final))
eco_final.head()

Rows ready to load: 2160


,country_id,time_id,GDP,Population,Access_to_Electricity,Crude_Oil_Price
0,56,1,1.913422e+10,33831764.0,71.5,50.750000
1,56,2,1.811657e+10,34700612.0,97.7,42.808333
2,56,3,1.875346e+10,35688935.0,97.7,52.816667
3,56,4,1.805322e+10,36743039.0,93.4,68.358333
4,56,5,1.879944e+10,37856121.0,97.7,61.416667


In [26]:
for index, row in eco_final.iterrows():
    cursor.execute("""
        INSERT INTO fact_economic 
        (country_id, time_id, gdp, population, access_to_electricity, crude_oil_price)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        int(row["country_id"]),
        int(row["time_id"]),
        float(row["GDP"]) if pd.notna(row["GDP"]) else None,
        float(row["Population"]) if pd.notna(row["Population"]) else None,
        float(row["Access_to_Electricity"]) if pd.notna(row["Access_to_Electricity"]) else None,
        float(row["Crude_Oil_Price"]) if pd.notna(row["Crude_Oil_Price"]) else None
    ))

conn.commit()
print("fact_economic loaded successfully")

fact_economic loaded successfully


In [27]:
cursor.execute("SELECT COUNT(*) FROM fact_economic")
print("Total rows in fact_economic:", cursor.fetchone()[0])

Total rows in fact_economic: 2160


In [28]:
# Create company map
cursor.execute("SELECT company_id, company_name FROM dim_company")
company_map = {name: id for id, name in cursor.fetchall()}

fin = company_fin.copy()
fin["company_id"] = fin["Company"].map(company_map)
fin["time_id"] = fin["Year"].map(time_map)

fin_final = fin[[
    "company_id", "time_id", "Revenue", "Gross Profit", "Operating Income",
    "Net Income", "EBITDA", "Basic EPS", "Diluted EPS"
]].dropna(subset=["company_id", "time_id"])

print("Rows ready to load:", len(fin_final))
fin_final.head()

Rows ready to load: 118


,company_id,time_id,Revenue,Gross Profit,Operating Income,Net Income,EBITDA,Basic EPS,Diluted EPS
0,1,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,8,8.146200e+10,2.085300e+10,1.383200e+10,1.258300e+10,1.765700e+10,4.020000,3.62
2,1,9,9.677300e+10,1.766000e+10,8.891000e+09,1.499900e+10,1.479600e+10,4.725697,4.31
3,1,10,9.769000e+10,1.745000e+10,7.760000e+09,7.130000e+09,1.470800e+10,2.230000,2.04
4,2,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
for index, row in fin_final.iterrows():
    cursor.execute("""
        INSERT INTO fact_company_financials 
        (company_id, time_id, revenue, gross_profit, operating_income, 
         net_income, ebitda, basic_eps, diluted_eps)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        int(row["company_id"]),
        int(row["time_id"]),
        float(row["Revenue"]) if pd.notna(row["Revenue"]) else None,
        float(row["Gross Profit"]) if pd.notna(row["Gross Profit"]) else None,
        float(row["Operating Income"]) if pd.notna(row["Operating Income"]) else None,
        float(row["Net Income"]) if pd.notna(row["Net Income"]) else None,
        float(row["EBITDA"]) if pd.notna(row["EBITDA"]) else None,
        float(row["Basic EPS"]) if pd.notna(row["Basic EPS"]) else None,
        float(row["Diluted EPS"]) if pd.notna(row["Diluted EPS"]) else None
    ))

conn.commit()
print("fact_company_financials loaded successfully")

fact_company_financials loaded successfully


In [30]:
cursor.execute("SELECT COUNT(*) FROM fact_company_financials")
print("Total rows in fact_company_financials:", cursor.fetchone()[0])

Total rows in fact_company_financials: 118


In [31]:
stock["company_id"] = stock["Company"].map(company_map)
stock["Date"] = pd.to_datetime(stock["Date"])

stock_final = stock[[
    "company_id", "Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"
]].dropna(subset=["company_id"])

print("Rows ready to load:", len(stock_final))
stock_final.head()

Rows ready to load: 71651


,company_id,Date,Open,High,Low,Close,Adj Close,Volume
0,12,2015-01-01,25.75,26.174999,25.575001,26.000000,21.183241,19139868
1,12,2015-01-02,26.25,26.775000,26.125000,26.700001,21.753563,37322834
2,12,2015-01-05,27.50,29.125000,27.275000,28.900000,23.545986,99572192
3,12,2015-01-06,28.50,28.725000,27.875000,28.275000,23.036777,74056464
4,12,2015-01-07,28.40,30.125000,28.250000,29.825001,24.299623,71420438


In [32]:
for index, row in stock_final.iterrows():
    cursor.execute("""
        INSERT INTO fact_stock_prices 
        (company_id, date, open, high, low, close, adj_close, volume)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        int(row["company_id"]),
        row["Date"].date(),
        float(row["Open"]),
        float(row["High"]),
        float(row["Low"]),
        float(row["Close"]),
        float(row["Adj Close"]),
        int(row["Volume"])
    ))

conn.commit()
print("fact_stock_prices loaded successfully")

fact_stock_prices loaded successfully


In [33]:
cursor.execute("SELECT COUNT(*) FROM fact_stock_prices")
print("Total rows in fact_stock_prices:", cursor.fetchone()[0])

Total rows in fact_stock_prices: 71651
